In [7]:
import yaml
import torch
import matplotlib.pyplot as plt
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer

c:\Users\91886\miniconda3\envs\awq\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [2]:
class GPT2_Model:
    def __init__(self, config):
        self.config = config
    
    def model(self):
        self.configuration = GPT2Config(**self.config)
        self.configuration._attn_implementation = "eager"
        self.model_ = GPT2LMHeadModel(self.configuration)

        return self.model_

    def print_param(self, model):
        
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total_params:,}")

        fp32_bytes = total_params * 4 
        fp16_bytes = total_params * 2

        print(f"FP32 memory: {fp32_bytes / 1024**2:.2f} MB")
        print(f"FP16 memory: {fp16_bytes / 1024**2:.2f} MB")

        param_memory = fp32_bytes/1024**2
        return param_memory

In [3]:
def get_register_hook(module):
    def get_forward_hook(name):
        def forward_hook_fun(module, inp, out):
            if isinstance(module, torch.nn.Embedding):
                x = inp[0]
                print('Embedding: ',name,'has shape: ',out.shape)
            if 'c_attn' in name:
                print('Attention: ',name,'has shape: ',out.reshape((out.shape[0],3,out.shape[1],out.shape[2]//3)).shape)
            if 'act' in name:
                print("Activation dim:", out.shape)
            if 'ln_f' in name:
                print("LayerNorm dim:", out.shape)
        return forward_hook_fun
    
    for name, layer in module.named_modules():
        layer.register_forward_hook(get_forward_hook(name))

In [8]:
data_path = '..\DATA\data.txt'
with open(data_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2", local_files_only=True)

# Use GPU if available, else fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
#GPT2 Small Model
print("\nGPT2 Small:")
with open("..\config\gpt2_large.yaml", "r") as f:
    config_small = yaml.safe_load(f)

gpt2_small = GPT2_Model(config_small)
gpt2_small_model = gpt2_small.model()
gpt2_small_model = gpt2_small_model.to(device)

inputs = tokenizer(raw_text, return_tensors="pt").to(device)

get_register_hook(gpt2_small_model)

input_test = inputs.input_ids[0][:1024]
with torch.no_grad():
    outputs = gpt2_small_model(input_test.unsqueeze(0), labels=input_test.unsqueeze(0), output_attentions=True)

print("Attention scores dim:", outputs.attentions[0].shape)

small_act_memory=outputs.attentions[0].shape[0] * outputs.attentions[0].shape[1] * outputs.attentions[0].shape[2] * outputs.attentions[0].shape[3] * 4/1024**2

#small_act_memory=max(act_memory)
print("Peak activation memory:",small_act_memory, "MB")

small_param_memory = gpt2_small.print_param(gpt2_small_model)

# Free Small model before loading Medium
del gpt2_small_model, gpt2_small, outputs, inputs
torch.cuda.empty_cache()


GPT2 Small:


: 

In [ ]:

#GPT2 Medium Model
print("\nGPT2 Medium:")
with open("gpt2_medium.yaml", "r") as f:
    config_medium = yaml.safe_load(f)

gpt2_medium = GPT2_Model(config_medium)
gpt2_medium_model = gpt2_medium.model().to(device)

inputs = tokenizer(raw_text, return_tensors="pt").to(device)

get_register_hook(gpt2_medium_model)

input_test = inputs.input_ids[0][:1024]
with torch.no_grad():
    outputs = gpt2_medium_model(input_test.unsqueeze(0), labels=input_test.unsqueeze(0), output_attentions=True)

print("Attention scores dim:", outputs.attentions[0].shape)

# act_memory=[]
# for idx in range(len(outputs.attentions)):
#     mem = outputs.attentions[idx].shape[0] * outputs.attentions[idx].shape[1] * outputs.attentions[idx].shape[2] * outputs.attentions[idx].shape[3] * 4/1024**2
#     print("Memory Bottleneck", mem, "MB")
#     act_memory.append(mem)

#medium_act_memory=max(act_memory)

medium_act_memory=outputs.attentions[0].shape[0] * outputs.attentions[0].shape[1] * outputs.attentions[0].shape[2] * outputs.attentions[0].shape[3] * 4/1024**2
print("Peak activation memory:",medium_act_memory, "MB")

medium_param_memory = gpt2_medium.print_param(gpt2_medium_model)

# Free Medium model before loading Large
del gpt2_medium_model, gpt2_medium, outputs, inputs
torch.cuda.empty_cache()

In [ ]:
#GPT2 Large Model
print("\nGPT2 Large:")
with open("gpt2_large.yaml", "r") as f:
    config_large = yaml.safe_load(f)

gpt2_large = GPT2_Model(config_large)
gpt2_large_model = gpt2_large.model().to(device)

inputs = tokenizer(raw_text, return_tensors="pt").to(device)

get_register_hook(gpt2_large_model)

input_test = inputs.input_ids[0][:1024]
with torch.no_grad():
    outputs = gpt2_large_model(input_test.unsqueeze(0), labels=input_test.unsqueeze(0), output_attentions=True)

print("Attention scores dim:", outputs.attentions[0].shape)

# act_memory=[]
# for idx in range(len(outputs.attentions)):
#     mem = outputs.attentions[idx].shape[0] * outputs.attentions[idx].shape[1] * outputs.attentions[idx].shape[2] * outputs.attentions[idx].shape[3] * 4/1024**2
#     print("Memory Bottleneck", mem, "MB")
#     act_memory.append(mem)

#large_act_memory=max(act_memory)
large_act_memory=outputs.attentions[0].shape[0] * outputs.attentions[0].shape[1] * outputs.attentions[0].shape[2] * outputs.attentions[0].shape[3] * 4/1024**2
print("Peak activation memory:",large_act_memory, "MB")

large_param_memory = gpt2_large.print_param(gpt2_large_model)

In [ ]:
print("Ratio act/param memory for GPT2 Small:", small_act_memory/small_param_memory)
print("Ratio act/param memory for GPT2 Medium:", medium_act_memory/medium_param_memory)
print("Ratio act/param memory for GPT2 Large:", large_act_memory/large_param_memory)

In [ ]:
if small_act_memory/small_param_memory > medium_act_memory/medium_param_memory and small_act_memory/small_param_memory > large_act_memory/large_param_memory:
    print("GPT2 Small has negligible activation")
elif medium_act_memory/medium_param_memory > small_act_memory/small_param_memory and medium_act_memory/medium_param_memory > large_act_memory/large_param_memory:
    print("GPT2 Medium has negligible activation")
else:    
    print("GPT2 Large has negligible activation")

In [ ]:
param_counts = [small_param_memory, medium_param_memory, large_param_memory]
fp16_memories = [small_param_memory, medium_param_memory, large_param_memory]
plt.figure(figsize=(8, 6))
plt.loglog(param_counts, fp16_memories, marker='o')
plt.title('Parameters Count vs FP16 Memory (Log-Log Scale)')
plt.xlabel('Parameters Count (MB)')
plt.ylabel('FP16 Memory (MB)')
plt.grid(True, which="both", ls="--")
plt.show()


print('Completed!')